# 3 · Build the private statistical database

Run in a **Colab CPU runtime**; no installation or GPU is needed. Upload one or more analysis ZIPs from notebook 2. To continue an existing database, also upload the previous database ZIP or its `voice_stat.sqlite` file. Download the resulting ZIP after importing.

Each analysis replaces the earlier dataset for the same **source-audio hash + actor + character**, so reruns do not duplicate observations. Supply only the newest analysis of each target/episode in one upload. Different encodings of the same episode have different source hashes: avoid importing both. The database stores transcripts, annotations, timing, model settings, and provenance; it does not contain audio or speaker embeddings.

The basic unit is a merged clip, not a sentence. Counts depend on the gap tolerance and other extraction settings. These statistics describe the **sampled performances**, not the actor's personality or complete career. Choose varied roles/episodes and verify speaker assignments before drawing broader conclusions. A result like 500 angry clips among 510 clips is about 98.04%, not 99%.

Automatic emotion scores are estimates. Unknown/mixed labels remain visible, and percentages use all eligible clips as the denominator. Speech-duration percentages use detected target speech, excluding the silence between turns. Manual voice attributes can coexist, so their percentages need not add to 100%; each uses only explicitly rated clips. Annotators and attribute definitions should be consistent across comparable datasets.

Generated databases and reports are private. Only publish notebooks after clearing outputs and removing personal inputs. Research is not a blanket copyright exemption: for example, [U.S. fair use](https://www.copyright.gov/fair-use/) is evaluated case by case. Applicable rules also depend on your jurisdiction and source/platform terms.

In [ ]:
ANALYSIS_ZIPS: list[str] = []     # Empty opens upload. May include the previous database ZIP too.
DATABASE_FILE: str = ""    # Optional standalone .sqlite already in Colab.
EXCLUDE_OVERLAPPING_CLIPS: bool = True  # Default statistics exclude contaminated clips.
REQUIRE_REVIEWED_IDENTITY: bool = False # True limits the main table to identity-accepted clips.

In [ ]:
import hashlib
import json
import shutil
import sqlite3
import uuid
import zipfile
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath
from typing import Any, TypeAlias

import pandas as pd
from google.colab import files
from IPython.display import display

JsonObject: TypeAlias = dict[str, Any]

work_root: Path = Path("/content/voice-stat-database") / uuid.uuid4().hex[:12]
work_root.mkdir(parents=True)
report_root: Path = work_root / "database"
report_root.mkdir()
database_path: Path = report_root / "voice_stat.sqlite"


def write_json(path: Path, value: JsonObject) -> None:
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")


if not ANALYSIS_ZIPS and not DATABASE_FILE:
    uploaded = files.upload()
    ANALYSIS_ZIPS: list[str] = [name for name in uploaded if Path(name).suffix.lower() == ".zip"]
    database_files = [name for name in uploaded if Path(name).suffix.lower() == ".sqlite"]
    if database_files:
        if DATABASE_FILE or len(database_files) != 1:
            raise ValueError("Choose only one existing database.")
        DATABASE_FILE: str = database_files[0]
    del uploaded

analysis_archives: list[tuple[str, str, JsonObject]] = []
database_archives: list[tuple[str, str]] = []
for filename in ANALYSIS_ZIPS:
    with zipfile.ZipFile(filename) as archive:
        manifest_names = [name for name in archive.namelist() if PurePosixPath(name).name == "manifest.json"]
        if len(manifest_names) != 1:
            raise ValueError(f"Expected one manifest in {filename}.")
        manifest_name = manifest_names[0]
        manifest: JsonObject = json.loads(archive.read(manifest_name))
        if manifest.get("schema_version") != 1:
            raise ValueError(f"Unsupported schema in {filename}.")
        if manifest["kind"] == "voice_stat_analysis":
            analysis_archives.append((filename, manifest_name, manifest))
        elif manifest["kind"] == "voice_stat_database":
            database_archives.append((filename, manifest_name))
        else:
            raise ValueError(f"{filename} is not an analysis or database ZIP.")

if len(database_archives) + bool(DATABASE_FILE) > 1:
    raise ValueError("Upload at most one existing database, standalone or ZIP.")
if DATABASE_FILE:
    shutil.copyfile(DATABASE_FILE, database_path)
elif database_archives:
    filename, manifest_name = database_archives[0]
    member = str(PurePosixPath(manifest_name).parent / "voice_stat.sqlite")
    with zipfile.ZipFile(filename) as archive:
        with archive.open(member) as source, database_path.open("wb") as destination:
            shutil.copyfileobj(source, destination)

existing_database = database_path.exists()
connection: sqlite3.Connection = sqlite3.connect(database_path)
connection.execute("PRAGMA foreign_keys = ON")
if existing_database and connection.execute("PRAGMA user_version").fetchone()[0] != 1:
    connection.close()
    raise ValueError("This database is not a supported voice-stat database.")
connection.executescript("""
CREATE TABLE IF NOT EXISTS datasets (
    dataset_id TEXT PRIMARY KEY,
    source_sha256 TEXT NOT NULL,
    actor_id TEXT NOT NULL,
    character_id TEXT NOT NULL,
    series_id TEXT NOT NULL,
    episode_id TEXT NOT NULL,
    manifest_json TEXT NOT NULL,
    imported_utc TEXT NOT NULL
);
CREATE TABLE IF NOT EXISTS observations (
    observation_id TEXT PRIMARY KEY,
    dataset_id TEXT NOT NULL REFERENCES datasets(dataset_id) ON DELETE CASCADE,
    actor_id TEXT NOT NULL,
    character_id TEXT NOT NULL,
    series_id TEXT NOT NULL,
    episode_id TEXT NOT NULL,
    start_seconds REAL NOT NULL,
    end_seconds REAL NOT NULL,
    speech_seconds REAL NOT NULL,
    transcript TEXT,
    automatic_emotion TEXT NOT NULL,
    manual_emotion TEXT,
    effective_emotion TEXT NOT NULL,
    emotion_source TEXT NOT NULL,
    identity_status TEXT NOT NULL,
    analysis_complete INTEGER NOT NULL,
    payload_json TEXT NOT NULL
);
CREATE INDEX IF NOT EXISTS observations_actor ON observations(actor_id);
PRAGMA user_version = 1;
""")

imported_ids: set[str] = set()
for filename, manifest_name, manifest in analysis_archives:
    dataset_key: list[str] = [manifest["source_sha256"], manifest["actor_id"], manifest["character_id"]]
    dataset_id = hashlib.sha256(json.dumps(dataset_key).encode()).hexdigest()[:24]
    if dataset_id in imported_ids:
        raise ValueError("Two supplied analyses describe the same episode/target. Upload only the newest one.")
    imported_ids.add(dataset_id)
    member = str(PurePosixPath(manifest_name).parent / manifest["analysis_file"])
    with zipfile.ZipFile(filename) as archive:
        rows: list[JsonObject] = [json.loads(line) for line in archive.read(member).decode("utf-8").splitlines() if line.strip()]
    if len(rows) != manifest["segment_count"]:
        raise ValueError(f"Segment count does not match the manifest in {filename}.")
    for row in rows:
        if [row["source_sha256"], row["actor_id"], row["character_id"]] != dataset_key:
            raise ValueError(f"An observation does not belong to its declared dataset in {filename}.")
    with connection:
        connection.execute("DELETE FROM datasets WHERE dataset_id = ?", (dataset_id,))
        connection.execute("INSERT INTO datasets VALUES (?, ?, ?, ?, ?, ?, ?, ?)", (
            dataset_id, *dataset_key, manifest["series_id"], manifest["episode_id"],
            json.dumps(manifest, ensure_ascii=False), datetime.now(timezone.utc).isoformat(),
        ))
        connection.executemany("INSERT INTO observations VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)", [
            (row["observation_id"], dataset_id, row["actor_id"], row["character_id"],
             row["series_id"], row["episode_id"], row["start_seconds"], row["end_seconds"],
             row["speech_seconds"], row["transcript"], row["automatic_emotion"], row["manual_emotion"],
             row["effective_emotion"], row["emotion_source"], row["identity_status"],
             int(row["analysis_complete"]), json.dumps(row, ensure_ascii=False, allow_nan=False)) for row in rows
        ])
    print(f"Imported {manifest['series_id']} / {manifest['episode_id']} / {manifest['character_id']}: {len(rows)} clips.")

stored_rows: list[JsonObject] = [
    json.loads(payload) for payload, in connection.execute(
        "SELECT payload_json FROM observations ORDER BY actor_id, series_id, episode_id, start_seconds"
    )
]
dataset_manifests: list[JsonObject] = [json.loads(payload) for payload, in connection.execute("SELECT manifest_json FROM datasets ORDER BY actor_id, series_id, episode_id")]
if not stored_rows:
    raise ValueError("Upload at least one analysis ZIP, or an existing populated database.")
frame: pd.DataFrame = pd.DataFrame(stored_rows)
print(f"Database: {len(dataset_manifests)} episode/target datasets; {len(frame)} clips.")

## Inspect coverage and summarize

`emotion_actor.csv` pools the sampled clips for each actor. `emotion_role_episode.csv` retains the actor/series/character/episode grouping. Matching `*_reviewed.csv` files require verified identity **and** a human emotion label. The all-clips reports can combine model predictions with manual corrections; `model_clips` and `human_clips` expose their sources.

`attributes.csv` reports only human attributes, grouped by definition hash and identity review status. Missing ratings are counted separately. Inspect `coverage.csv` and `provenance.csv` for pending clips, exclusions, model revisions, and incompatible extraction settings before comparing totals.

In [ ]:
coverage_rows: list[JsonObject] = []
for keys, group in frame.groupby(["actor_id", "series_id", "character_id", "episode_id"], dropna=False):
    coverage_rows.append({
        **dict(zip(["actor_id", "series_id", "character_id", "episode_id"], keys)),
        "total_clips": len(group), "complete_clips": int(group["analysis_complete"].sum()),
        "pending_clips": int((~group["analysis_complete"]).sum()),
        "identity_accepted_clips": int((group["identity_status"] == "accepted").sum()),
        "identity_unreviewed_clips": int((group["identity_status"] == "unreviewed").sum()),
        "identity_rejected_clips": int((group["identity_status"] == "rejected").sum()),
        "overlapping_clips": int((group["overlap_seconds"] > 0).sum()),
        "human_emotion_clips": int(group["manual_emotion"].notna().sum()),
        "unknown_effective_emotion_clips": int((group["effective_emotion"] == "unknown").sum()),
    })
coverage: pd.DataFrame = pd.DataFrame(coverage_rows)
coverage.to_csv(report_root / "coverage.csv", index=False)
display(coverage)

eligible: pd.DataFrame = frame[frame["analysis_complete"] & (frame["identity_status"] != "rejected")].copy()
if EXCLUDE_OVERLAPPING_CLIPS:
    eligible: pd.DataFrame = eligible[eligible["overlap_seconds"] == 0].copy()
if REQUIRE_REVIEWED_IDENTITY:
    eligible: pd.DataFrame = eligible[eligible["identity_status"] == "accepted"].copy()
reviewed: pd.DataFrame = eligible[(eligible["identity_status"] == "accepted") & eligible["manual_emotion"].notna()].copy()


def summarize_emotions(data: pd.DataFrame, group_columns: list[str]) -> pd.DataFrame:
    columns = group_columns + ["emotion", "clips", "total_clips", "clip_percent",
                               "speech_seconds", "total_speech_seconds", "speech_percent",
                               "human_clips", "model_clips", "identity_unreviewed_clips"]
    summary_rows: list[JsonObject] = []
    for keys, group in data.groupby(group_columns, dropna=False):
        keys = keys if isinstance(keys, tuple) else (keys,)
        total_seconds = float(group["speech_seconds"].sum())
        for label, subset in group.groupby("effective_emotion", dropna=False):
            seconds = float(subset["speech_seconds"].sum())
            summary_rows.append({
                **dict(zip(group_columns, keys)), "emotion": label,
                "clips": len(subset), "total_clips": len(group),
                "clip_percent": 100 * len(subset) / len(group),
                "speech_seconds": seconds, "total_speech_seconds": total_seconds,
                "speech_percent": 100 * seconds / total_seconds if total_seconds else None,
                "human_clips": int((subset["emotion_source"] == "human").sum()),
                "model_clips": int((subset["emotion_source"] == "model").sum()),
                "identity_unreviewed_clips": int((subset["identity_status"] == "unreviewed").sum()),
            })
    return pd.DataFrame(summary_rows, columns=columns)


actor_summary: pd.DataFrame = summarize_emotions(eligible, ["actor_id"])
episode_columns: list[str] = ["actor_id", "series_id", "character_id", "episode_id"]
episode_summary: pd.DataFrame = summarize_emotions(eligible, episode_columns)
actor_reviewed: pd.DataFrame = summarize_emotions(reviewed, ["actor_id"])
episode_reviewed: pd.DataFrame = summarize_emotions(reviewed, episode_columns)
for name, table in {
    "emotion_actor.csv": actor_summary, "emotion_role_episode.csv": episode_summary,
    "emotion_actor_reviewed.csv": actor_reviewed, "emotion_role_episode_reviewed.csv": episode_reviewed,
}.items():
    table.to_csv(report_root / name, index=False)
display(actor_summary)
print("Clips requiring both verified identity and human emotion:", len(reviewed))
display(actor_reviewed)

attribute_rows: list[JsonObject] = []
for keys, group in eligible.groupby(["actor_id", "annotation_schema", "identity_status"], dropna=False):
    actor, schema, identity_status = keys
    names = sorted({name for attributes in group["attributes"] for name in attributes})
    for name in names:
        rated = [row for row in group.to_dict("records") if row["attributes"].get(name) in (0, 1)]
        present = [row for row in rated if row["attributes"][name] == 1]
        rated_seconds = sum(row["speech_seconds"] for row in rated)
        present_seconds = sum(row["speech_seconds"] for row in present)
        definition = group.iloc[0]["attribute_definitions"].get(name)
        attribute_rows.append({
            "actor_id": actor, "annotation_schema": schema, "identity_status": identity_status,
            "attribute": name, "definition": definition,
            "rated_clips": len(rated), "present_clips": len(present),
            "unrated_clips": len(group) - len(rated),
            "percent_of_rated_clips": 100 * len(present) / len(rated) if rated else None,
            "rated_speech_seconds": rated_seconds, "present_speech_seconds": present_seconds,
            "percent_of_rated_speech": 100 * present_seconds / rated_seconds if rated_seconds else None,
        })
attributes: pd.DataFrame = pd.DataFrame(attribute_rows, columns=[
    "actor_id", "annotation_schema", "identity_status", "attribute", "definition",
    "rated_clips", "present_clips", "unrated_clips", "percent_of_rated_clips",
    "rated_speech_seconds", "present_speech_seconds", "percent_of_rated_speech",
])
attributes.to_csv(report_root / "attributes.csv", index=False)
display(attributes)

provenance_rows: list[JsonObject] = []
for manifest in dataset_manifests:
    provenance_rows.append({
        **{key: manifest[key] for key in ["actor_id", "character_id", "series_id", "episode_id", "source_sha256"]},
        "extraction_settings": json.dumps(manifest["settings"], sort_keys=True),
        "extraction_models": json.dumps(manifest["models"], sort_keys=True),
        "analysis_settings": json.dumps(manifest["analysis_settings"], sort_keys=True),
    })
provenance: pd.DataFrame = pd.DataFrame(provenance_rows)
provenance.to_csv(report_root / "provenance.csv", index=False)
if provenance["extraction_settings"].nunique() > 1:
    print("Extraction settings differ across datasets. Check provenance.csv before comparing clip counts.")
print(f"Main summaries use {len(eligible)} clips; {len(frame) - len(eligible)} excluded by completeness, identity, or overlap rules.")

In [ ]:
report_manifest: JsonObject = {
    "schema_version": 1, "kind": "voice_stat_database",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "database_file": "voice_stat.sqlite", "dataset_count": len(dataset_manifests),
    "observation_count": len(frame), "eligible_observation_count": len(eligible),
    "reviewed_observation_count": len(reviewed),
    "settings": {"exclude_overlapping_clips": EXCLUDE_OVERLAPPING_CLIPS,
                 "require_reviewed_identity": REQUIRE_REVIEWED_IDENTITY},
    "clip_unit": "target turns merged by the configured gap tolerance",
    "emotion_denominator": "all eligible clips, including unknown and mixed",
    "duration_denominator": "detected target speech seconds in eligible clips",
    "attribute_denominator": "clips explicitly rated for each attribute and definition schema",
}
write_json(report_root / "manifest.json", report_manifest)
connection.commit()
connection.close()
archive_path: Path = work_root / "voice_stat_database.zip"
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(report_root.iterdir()):
        if path.is_file():
            archive.write(path, arcname=str(Path("database") / path.name))
files.download(str(archive_path))
print("Keep this private ZIP. Upload it with new analysis ZIPs next time to continue your database.")